# Entrenar el detector especialitzat de llegenda

Aquest notebook **no torna a entrenar les columnes**. Reutilitza els JSON del model `20261107-151404`, aplica el `ymin/ymax` compartit i entrena un YOLO d'una sola classe per a `lithology_legend`.

Abans de començar:

1. Genera el paquet actual amb `python3 tools/package_colab.py`.
2. Puja `artifacts/colab_bundle.zip` a `My Drive/subsurface-bboxes/colab_bundle.zip`.
3. Conserva a Drive el run anterior `runs/20261107-151404`, especialment `validation/predictions/`.
4. Activa una GPU T4 o superior a Colab.

El split continua sent per família: 41 documents de train i 10 de validació.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from datetime import datetime
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/subsurface-bboxes')
BUNDLE_PATH = DRIVE_ROOT / 'colab_bundle.zip'
WORKDIR = Path('/content/subsurface-bboxes')
DATASET = WORKDIR / 'dataset'
BASE_RUN_DIR = DRIVE_ROOT / 'runs' / '20261107-151404'
LEGEND_RUN_DIR = DRIVE_ROOT / 'legend-runs' / datetime.now().strftime('%Y%m%d-%H%M%S')
AUDIT = LEGEND_RUN_DIR / 'audit'
LEGEND_DATASET = LEGEND_RUN_DIR / 'dataset'

IMGSZ = 2048
TRAIN_VARIANTS = 4
EPOCHS = 80
PATIENCE = 15
BATCH = 2
CONFIDENCES = [0.05, 0.10, 0.15]
MERGE_FRAGMENTS = True  # Reconstrueix peces alineades de la mateixa llegenda.
# Per reutilitzar pesos entrenats, indica el seu Path de Drive; None entrena de nou.
EXISTING_LEGEND_WEIGHTS = None

LEGEND_RUN_DIR.mkdir(parents=True, exist_ok=True)
assert BUNDLE_PATH.exists(), f'No trobo {BUNDLE_PATH}'
assert BASE_RUN_DIR.exists(), f'No trobo el run base {BASE_RUN_DIR}'
print('Dataset:', DATASET)
print('Run base:', BASE_RUN_DIR)
print('Nou run:', LEGEND_RUN_DIR)


In [ ]:
import importlib
import os
import shutil
import subprocess
import sys
import zipfile

# Comença net per no barrejar el bundle nou amb fitxers d'una execució anterior.
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
WORKDIR.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE_PATH) as bundle:
    root = WORKDIR.resolve()
    for member in bundle.infolist():
        if not (WORKDIR / member.filename).resolve().is_relative_to(root):
            raise ValueError(f'Ruta invàlida al ZIP: {member.filename}')
    bundle.extractall(WORKDIR)

required_source = WORKDIR / 'bbox_detector' / 'legend_selection.py'
if not required_source.is_file():
    raise FileNotFoundError(
        f'El ZIP és antic: falta {required_source}. Torna a pujar artifacts/colab_bundle.zip.'
    )

os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))
# Si la sessió ja havia importat el projecte antic, elimina'l de la memòria.
for module_name in list(sys.modules):
    if module_name == 'bbox_detector' or module_name.startswith('bbox_detector.'):
        del sys.modules[module_name]
importlib.invalidate_caches()

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
assert DATASET.exists(), f'El paquet no conté {DATASET}'
import bbox_detector.legend_data
print('Entorn preparat:', WORKDIR)
print('Mòdul de llegenda:', bbox_detector.legend_data.__file__)


## 1. Auditar el dataset i crear els retalls de llegenda

Primer es reprodueix exactament el split per família. Després es creen retalls quadrats de tota l'amplada: quatre variants positives per document de train, una mostra negativa del cos, i només una positiva i una negativa per document de validació.


In [ ]:
import json
from bbox_detector.data import prepare_dataset
from bbox_detector.legend_data import prepare_legend_dataset

manifest = prepare_dataset(DATASET, AUDIT, materialize=False)
legend_manifest = None
if EXISTING_LEGEND_WEIGHTS is None:
    legend_manifest = prepare_legend_dataset(
        AUDIT / 'manifest.json',
        LEGEND_DATASET,
        imgsz=IMGSZ,
        train_variants=TRAIN_VARIANTS,
    )
print('Split original:', json.dumps(manifest['summary'], indent=2))
if legend_manifest is not None:
    print('Dataset de llegenda:', json.dumps(legend_manifest['summary'], indent=2))
    print('YAML:', LEGEND_DATASET / 'dataset.yaml')
else:
    print('Reutilitzant pesos; no es generen retalls d’entrenament.')


## 2. Comprovar visualment els retalls

Verd indica una mostra positiva amb la caixa YOLO; una mostra negativa no ha de contenir la llegenda. Si això no és correcte, atura't abans d'entrenar.


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

if legend_manifest is None:
    print('Sense previsualització de train: s’utilitzen pesos existents.')
else:
    samples = legend_manifest['samples']
    examples = [next(s for s in samples if s['split'] == 'train' and s['kind'] == kind) for kind in ('positive', 'negative')]
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    for axis, sample in zip(axes, examples):
        image = Image.open(sample['image_path']).convert('RGB')
        label = Path(sample['label_path']).read_text().strip()
        if label:
            _, cx, cy, bw, bh = map(float, label.split())
            w, h = image.size
            ImageDraw.Draw(image).rectangle(((cx-bw/2)*w, (cy-bh/2)*h, (cx+bw/2)*w, (cy+bh/2)*h), outline='lime', width=5)
        axis.imshow(image)
        axis.set_title(f"{sample['kind']}: {sample['image_name']}")
        axis.axis('off')
    plt.show()


## 3. Entrenar només la llegenda

YOLO11s comença de pesos preentrenats i aprèn una sola classe. Amb una T4, prova primer els valors per defecte. Si falta memòria, baixa `BATCH` a 1; no redueixis `IMGSZ` abans de provar-ho.


In [ ]:
import torch
from bbox_detector.legend_training import train_legend_model

DEVICE = '0' if torch.cuda.is_available() else 'cpu'
if EXISTING_LEGEND_WEIGHTS is not None:
    LEGEND_WEIGHTS = Path(EXISTING_LEGEND_WEIGHTS)
    assert LEGEND_WEIGHTS.is_file(), f'No trobo {LEGEND_WEIGHTS}'
    training = {'reused_weights': str(LEGEND_WEIGHTS)}
else:
    if DEVICE == 'cpu':
        raise RuntimeError('Activa GPU a Entorn d’execució > Canvia el tipus d’entorn > T4 GPU')

    training = train_legend_model(
        LEGEND_DATASET / 'dataset.yaml',
        LEGEND_RUN_DIR / 'training',
        model='yolo11s.pt',
        epochs=EPOCHS,
        patience=PATIENCE,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
    )
    LEGEND_WEIGHTS = Path(training['best_weights'])
print('Millors pesos:', LEGEND_WEIGHTS)
print(json.dumps(training, indent=2))


## 4. Comparar amb el model existent

Aquesta és la cel·la de decisió. No reexecuta el detector de columnes: carrega els seus JSON, comparteix `ymin/ymax` entre les dues columnes i compara la llegenda antiga amb el nou especialista a tres llindars de confiança. El guany real es mesura sempre sobre els 10 documents de validació.

Amb `MERGE_FRAGMENTS=True`, uneix candidats connectats i alineats verticalment. El baseline també participa en la selecció: es conserva si cap confiança el supera, inclosos els empats. Aquesta selecció utilitza validació; no és un resultat de test independent.


In [ ]:
import copy
from bbox_detector.common import write_json
from bbox_detector.evaluation import evaluate_manifest
from bbox_detector.inference import share_column_vertical_extent
from bbox_detector.legend_inference import UltralyticsLegendDetector, fuse_legend_prediction
from bbox_detector.legend_selection import select_legend_strategy

candidates = [
    BASE_RUN_DIR / 'validation' / 'predictions',
    BASE_RUN_DIR / 'validation_shared_y' / 'predictions',
]
BASE_PREDICTIONS = next((path for path in candidates if path.is_dir()), None)
if BASE_PREDICTIONS is None:
    raise FileNotFoundError(f'No trobo els JSON base a: {candidates}')

def load_base(image_path):
    path = BASE_PREDICTIONS / f'{Path(image_path).stem}.json'
    payload = json.loads(path.read_text())
    return payload

def shared_y_predictor(image_path, **_kwargs):
    payload = copy.deepcopy(load_base(image_path))
    if share_column_vertical_extent(payload['bounding_boxes']) is None:
        raise ValueError(f'Falten columnes a {Path(image_path).name}')
    return payload

baseline = evaluate_manifest(
    AUDIT / 'manifest.json', None, LEGEND_RUN_DIR / 'baseline_shared_y',
    predictor=shared_y_predictor, save_previews=False,
)

legend_detector = UltralyticsLegendDetector(LEGEND_WEIGHTS)
reports = {}
for confidence in CONFIDENCES:
    def fused_predictor(image_path, _confidence=confidence, **_kwargs):
        return fuse_legend_prediction(
            image_path, load_base(image_path), detector=legend_detector,
            imgsz=IMGSZ, confidence=_confidence, device=DEVICE,
            merge_fragments=MERGE_FRAGMENTS,
        )
    reports[confidence] = evaluate_manifest(
        AUDIT / 'manifest.json', None,
        LEGEND_RUN_DIR / f'fused_conf_{confidence:.2f}',
        predictor=fused_predictor, imgsz=IMGSZ, conf=confidence,
        device=DEVICE, save_previews=False,
    )

def score(report):
    summary = report['summary']
    return (
        summary['all_regions_iou_ge_threshold']['documents'],
        summary['classes']['lithology_legend']['iou_mean'],
    )

selection = select_legend_strategy(baseline, reports)
selection['merge_fragments'] = MERGE_FRAGMENTS if selection['strategy'] != 'baseline' else False
BEST_CONFIDENCE = selection['confidence']
best_report = baseline if BEST_CONFIDENCE is None else reports[BEST_CONFIDENCE]
comparison = {
    'base_predictions': str(BASE_PREDICTIONS),
    'legend_weights': str(LEGEND_WEIGHTS),
    'baseline': baseline['summary'],
    'candidates': {str(conf): report['summary'] for conf, report in reports.items()},
    'merge_fragments': MERGE_FRAGMENTS,
    'best_confidence': BEST_CONFIDENCE,
    'selection': selection,
}
write_json(LEGEND_RUN_DIR / 'comparison.json', comparison)
print('Baseline:', score(baseline))
for confidence, report in reports.items():
    print(f'conf={confidence:.2f}:', score(report))
print('Estratègia seleccionada:', selection['strategy'])
print('Confidence seleccionada (None = baseline):', BEST_CONFIDENCE)
print('Millor confidence del especialista:', selection['best_specialist_confidence'])


## 5. Resultat final, comparacions visuals i exportació

Es torna a executar l’estratègia seleccionada (baseline o especialista) per generar imatges de comparació. La taula mostra l'IoU de llegenda abans/després i si cada document passa les tres regions amb IoU ≥ 0,8. Finalment es crea un ZIP portable al mateix directori de Drive.


In [ ]:
import pandas as pd
import shutil


def final_predictor(image_path, **_kwargs):
    if selection['strategy'] == 'baseline':
        return shared_y_predictor(image_path)
    return fuse_legend_prediction(
        image_path, load_base(image_path), detector=legend_detector,
        imgsz=IMGSZ, confidence=BEST_CONFIDENCE, device=DEVICE,
        merge_fragments=MERGE_FRAGMENTS,
    )

final_report = evaluate_manifest(
    AUDIT / 'manifest.json', None, LEGEND_RUN_DIR / 'final_best',
    predictor=final_predictor, imgsz=IMGSZ, conf=BEST_CONFIDENCE if BEST_CONFIDENCE is not None else 0.25,
    device=DEVICE, save_previews=True,
)

before = {doc['image_name']: doc for doc in baseline['documents']}
rows = []
for doc in final_report['documents']:
    old = before[doc['image_name']]
    rows.append({
        'document': doc['image_name'],
        'legend_IoU_abans': old['metrics']['regions']['lithology_legend']['iou'],
        'legend_IoU_despres': doc['metrics']['regions']['lithology_legend']['iou'],
        '3_regions_abans': old['metrics']['all_regions_iou_ge_threshold'],
        '3_regions_despres': doc['metrics']['all_regions_iou_ge_threshold'],
    })
display(pd.DataFrame(rows).sort_values('legend_IoU_despres', ascending=False))

EXPORT = LEGEND_RUN_DIR / 'export'
EXPORT.mkdir(exist_ok=True)
shutil.copy2(LEGEND_WEIGHTS, EXPORT / 'legend_best.pt')
shutil.copy2(LEGEND_RUN_DIR / 'comparison.json', EXPORT / 'comparison.json')
shutil.copy2(LEGEND_RUN_DIR / 'final_best' / 'report.json', EXPORT / 'validation_report.json')
shutil.copy2(WORKDIR / 'requirements.txt', EXPORT / 'requirements.txt')
shutil.copytree(WORKDIR / 'bbox_detector', EXPORT / 'bbox_detector', dirs_exist_ok=True, ignore=shutil.ignore_patterns('__pycache__'))
archive = shutil.make_archive(str(LEGEND_RUN_DIR / 'legend_specialist'), 'zip', EXPORT)

summary = final_report['summary']
print('Estratègia final:', selection['strategy'])
print('Llegenda IoU mean:', summary['classes']['lithology_legend']['iou_mean'])
print('Documents amb les 3 IoU >= 0.8:', summary['all_regions_iou_ge_threshold'])
print('Comparacions:', LEGEND_RUN_DIR / 'final_best' / 'comparisons')
print('Paquet exportat:', archive)
